# QLoRA fine-tuning for code refinement — Kaggle runner (P100)

Kaggle-specific variant of `colab_train_clean.ipynb`. Differences from the Colab
version, and why each one is here:

* **Pinned to a single GPU.** Kaggle's "GPU P100 x2" option exposes **two** P100s.
  Without an explicit multi-GPU launch strategy (`accelerate launch --multi_gpu` +
  code changes for per-rank device placement), Hugging Face's `Trainer` detects
  the second GPU and silently wraps the model in legacy `torch.nn.DataParallel`.
  That is not compatible with `bitsandbytes` 4-bit quantized layers — it crashes
  with `RuntimeError: CUDA error: CUBLAS_STATUS_NOT_INITIALIZED` the first time a
  quantized layer runs on a replica thread. Setting `CUDA_VISIBLE_DEVICES=0`
  **before importing torch** hides the second GPU from the process entirely, so
  `Trainer` never attempts `DataParallel` and training runs on one clean P100.
  The second P100 sits idle under this notebook — see the note at the bottom on
  what real 2-GPU training would require.
* **Working directory is `/kaggle/working`**, the only writable, persisted path.
* **No `google.colab` calls** — outputs are just left in `/kaggle/working`; Kaggle's
  own "Output" tab lets you download them after the session ends, no zip-and-`files.download()`
  dance required (a zip is still built below for convenience).
* **Credentials** try Kaggle's Secrets add-on first, falling back to a prompt.

**Before running:** Notebook settings (right sidebar) → **Accelerator: GPU P100 x2**,
and **Internet: On** (needed to clone the repo and download the base model). Kaggle
GPU sessions have a **9-hour hard limit** and a **~30 GPU-hour/week quota** — keep
both in mind before committing to the full 3-epoch run.

**Run cell 1 first, alone, before anything else imports torch.**

## 1 — Pin to a single GPU (must run before any torch/transformers import)

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # hides the 2nd P100 so Trainer never tries DataParallel
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
print("CUDA_VISIBLE_DEVICES set to:", os.environ["CUDA_VISIBLE_DEVICES"])

## 2 — GPU check

Should show exactly **one** P100 to torch, even though `nvidia-smi` (which ignores
`CUDA_VISIBLE_DEVICES`) still lists both physical GPUs on the box.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch
print("torch sees", torch.cuda.device_count(), "GPU(s):", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.device_count() == 1, "Expected exactly 1 visible GPU — check cell 1 ran first.

## 3 — Clone the repo

In [ ]:
%cd /kaggle/working
!rm -rf loRA-code-refinement
!git clone -q https://github.com/hamnaraeel/loRA-code-refinement.git
%cd /kaggle/working/loRA-code-refinement
!ls -la data/processed data/benchmark

## 4 — Install

Same pinned combo as the Colab notebook: `transformers==4.46.3` / `peft==0.13.2` /
`trl==0.11.4` / `accelerate==1.2.1` — newer TRL drops the completion-only-loss
collator this project relies on for Mistral. `bitsandbytes` is left as a loose
`>=0.43.1` bound; exact-pinning it to older releases has broken against this
transformers version on other cloud images (`triton.ops` import error).

In [ ]:
!pip install -q "transformers==4.46.3" "peft==0.13.2" "trl==0.11.4" "accelerate==1.2.1" "tokenizers<0.21" \
    "bitsandbytes>=0.43.1" "datasets>=2.19" sacrebleu Levenshtein pyyaml pydantic typer rich matplotlib wandb
!pip install -q -e . --no-deps

import torch, transformers, peft, trl, accelerate, bitsandbytes
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "| visible GPUs:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0), "| vram_gb:", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1))
print("transformers", transformers.__version__, "| peft", peft.__version__, "| trl", trl.__version__, "| accelerate", accelerate.__version__)
print("bitsandbytes", bitsandbytes.__version__)

from trl import DataCollatorForCompletionOnlyLM  # must succeed — confirms the legacy masking path is active
print("legacy masking path: OK")

## 5 — Credentials (optional)

* **Hugging Face** — only needed for gated bases (e.g. Llama 3). Mistral-7B-Instruct-v0.3 is ungated.
* **Weights & Biases** — optional; the pipeline logs to `artifacts/runs/<name>/metrics.jsonl` regardless.

Tries Kaggle's **Add-ons → Secrets** first (add a secret named `HF_TOKEN` and/or
`WANDB_API_KEY` there), falling back to an interactive prompt.

In [ ]:
import os, getpass

def get_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

# hf_token = get_secret("HF_TOKEN")
# if hf_token:
#     from huggingface_hub import login; login(token=hf_token)

use_wandb = False  # set True to log to W&B
if use_wandb:
    key = get_secret("WANDB_API_KEY") or getpass.getpass("W&B API key: ")
    os.environ["WANDB_API_KEY"] = key
    os.environ["WANDB_PROJECT"] = "lora-code-refinement" 

## 6 — Sanity-check the masking fix before spending GPU time

Cheap (CPU, no model download): confirms the response-template token IDs actually
occur in the real training set. This was a real, confirmed bug in an earlier
version of `train.py` — the response template derived from a synthetic probe
tokenized differently than it did in real examples (which all start with a code
fence), so completion-only loss masking matched 0% of real training examples.
Fixed in this repo's `train.py`; this cell re-verifies it on your clone.

In [ ]:
import sys, json
sys.path.insert(0, "src")
from transformers import AutoTokenizer
from coderefine.train import find_response_template, render_dataset, _check_response_template_coverage

BASE = "mistralai/Mistral-7B-Instruct-v0.3"
tok = AutoTokenizer.from_pretrained(BASE)
template = find_response_template(tok)
print("response_template:", repr(template))

rows = [json.loads(l) for l in open("data/processed/train.jsonl")]
ds = render_dataset(rows, tok)
_check_response_template_coverage(tok, ds, template, sample_size=len(ds))
print("Masking coverage check passed — safe to train.")

## 7 — Measure real throughput before committing to the full run

GPU speed on shared cloud instances varies enough (T4 sessions on Colab have shown
up to ~10x spread) that trusting any a-priori estimate is a bad idea. This times a
tiny slice on **your actual P100** and extrapolates, so you know the real number
before spending hours or your weekly Kaggle GPU quota.

In [ ]:
import time
t0 = time.time()
!coderefine train configs/qlora_mistral7b.yaml --set train.max_train_samples=64 --set train.num_epochs=1 --set name=speedtest
elapsed = time.time() - t0
print(f"\n{elapsed:.0f}s for a 64-example / 1-epoch timing run.")

# effective_batch = per_device_batch_size(2) * grad_accum(8) = 16 -> ceil(64/16) = 4 steps in this timing run
steps_this_run = 4
full_run_steps = 375  # 2000 examples, effective batch 16, 3 epochs -> ceil(2000/16)*3
projected_hours = (elapsed / steps_this_run) * full_run_steps / 3600
print(f"Projected full 3-epoch run at this rate: ~{projected_hours:.1f} hours")
print(f"Projected 1-epoch run at this rate: ~{projected_hours/3:.1f} hours")

## 8 — Train

Pick full (3 epochs, matches the README's headline numbers) or the faster 1-epoch
run based on what step 7 projected.

In [ ]:
# Full run (3 epochs):
!coderefine train configs/qlora_mistral7b.yaml

# Faster alternative — uncomment to use instead if step 7 projected too long:
# !coderefine train configs/qlora_mistral7b.yaml --set train.num_epochs=1

In [ ]:
# Loss curves from the local metric log (works with or without W&B)
import json, pathlib
import matplotlib.pyplot as plt

run = "qlora-mistral7b-r16"
rows = [json.loads(l) for l in (pathlib.Path("artifacts/runs")/run/"metrics.jsonl").read_text().splitlines() if l.strip()]
tr = [(r["_step"], r["loss"]) for r in rows if "loss" in r]
ev = [(r["_step"], r["eval_loss"]) for r in rows if "eval_loss" in r]

fig, ax = plt.subplots(figsize=(8, 4.5))
if tr: ax.plot(*zip(*tr), label="train loss", lw=1.4)
if ev: ax.plot(*zip(*ev), label="eval loss", lw=1.4, marker="o")
ax.set_xlabel("step"); ax.set_ylabel("loss"); ax.legend(); ax.set_title(run)
plt.tight_layout(); plt.show()

## 9 — Evaluate

Base first — it is the denominator of every improvement claim — then the tuned
model on the identical benchmark with the identical prompts and greedy decoding.

In [ ]:
ADAPTER = "artifacts/runs/qlora-mistral7b-r16/adapter"
BASE    = "mistralai/Mistral-7B-Instruct-v0.3"

!coderefine evaluate --split benchmark --base-model $BASE --tag base --load-in-4bit
!coderefine evaluate --split benchmark --base-model $BASE --adapter $ADAPTER --tag tuned --load-in-4bit
!coderefine compare artifacts/eval/base__benchmark.predictions.jsonl \
                    artifacts/eval/tuned__benchmark.predictions.jsonl

## 10 — Catastrophic forgetting check

In [ ]:
!coderefine forgetting --base-model $BASE --load-in-4bit
!coderefine forgetting --base-model $BASE --adapter $ADAPTER --load-in-4bit

## 11 — The sacred test split

Only run this once, after the configuration is frozen. Everything above used
validation and the benchmark.

In [ ]:
!coderefine evaluate --split test --final --base-model $BASE --tag base-test  --load-in-4bit
!coderefine evaluate --split test --final --base-model $BASE --adapter $ADAPTER --tag tuned-test --load-in-4bit
!coderefine compare artifacts/eval/base-test__test.predictions.jsonl \
                    artifacts/eval/tuned-test__test.predictions.jsonl \
                    --out-path artifacts/eval/comparison_test.json

## 12 — Report, package for download

Kaggle persists everything under `/kaggle/working` to the notebook's **Output** tab
automatically after the session ends — no explicit download call needed — but the
zip below makes grabbing just the results in one file easier.

In [ ]:
!coderefine report
!coderefine export $ADAPTER --out-dir artifacts/release --base-model $BASE
from IPython.display import Markdown, display
display(Markdown(open("reports/EXPERIMENT_REPORT.md").read()))

In [ ]:
!zip -qr /kaggle/working/coderefine_results.zip artifacts/release artifacts/eval artifacts/runs artifacts/forgetting reports data/processed/dataset_card.json data/benchmark/benchmark_card.json
!du -h /kaggle/working/coderefine_results.zip
print("Saved to /kaggle/working/coderefine_results.zip — download it from the notebook's Output tab.")

## Note — using both P100s for real

This notebook deliberately uses one GPU to sidestep the `DataParallel` crash. Real
2-GPU speedup needs `accelerate launch --multi_gpu` (or `torchrun`) plus removing
the hardcoded `device_map={"": 0}` in `src/coderefine/train.py` in favor of a
per-process device map keyed to the local rank, and re-validating that
`bitsandbytes` 4-bit layers behave correctly under `DistributedDataParallel`
(generally safer than `DataParallel`, but still worth confirming empirically before
trusting the numbers from a run that used it). Ask if you want that wired up —
it's a real code change, not just a notebook change, so it's worth doing
deliberately rather than as a drive-by edit here.